# Assignment 2A — Prompting as System Design

**Group 172**

| Member | BITS ID | Contribution |
|---|---|---|
| Yeldona Desai V Vittal | 2025AE05316 | 100% |
| Naveen K. B | — | 100% |
| Naman Aryan | — | 100% |
| Patil Manaliben Dipakbhai | — | 100% |

**Reused from Assignment 1A:** domain corpus (13 cleaned source papers, Variant 1 — scientific research papers), instruction dataset, and the base model `HuggingFaceTB/SmolLM2-360M` with a continual-pre-training (CPT) + QLoRA fine-tuning pipeline.

**Generator for 2A:** the 1A pipeline (CPT on the domain corpus, then QLoRA instruction fine-tuning), greedy decoding, `max_new_tokens=200`. The original 1A adapter weights were lost (ephemeral Colab storage) and have been retrained here following 1A's own code faithfully, with one deviation noted and justified in Step 1 below.

**Judge for Part A:** `Qwen2.5-7B-Instruct`, deterministic decoding.

**Runtime:** this notebook was developed and validated on a local CPU machine at reduced (dev) scale; the full-fidelity retrain (10 CPT epochs, 8 SFT epochs, matching 1A's hyperparameters exactly) is run separately on the BITS Prayogshala lab machine, and the resulting adapter is substituted in before final submission.


## Setup — Colab bootstrap

Running this notebook via `colab.research.google.com/github/.../Assignment2A_Group172.ipynb` only loads this file — not the rest of the repo (`instruction_dataset.jsonl`, `domain_corpus/`, `benchmark_10.json`, `train_adapter.py`) that the later cells need on disk next to it. This cell detects Colab, clones the repo, and installs the GPU dependency set. On a local machine or a labs box where the repo is already cloned and the environment already set up (per `RUNBOOK.md`), it does nothing.


In [ ]:
import os
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/VijayYD/2025AE05316-LLM-2A.git"
    REPO_DIR = "2025AE05316-LLM-2A"
    if not os.path.isdir(REPO_DIR):
        print(f"Cloning {REPO_URL} ...")
        get_ipython().system(f"git clone {REPO_URL}")
    os.chdir(REPO_DIR)
    print("Installing dependencies from requirements-gpu.txt ...")
    get_ipython().system(f"{sys.executable} -m pip install -q -r requirements-gpu.txt")
    print("Colab bootstrap done. Working directory:", os.getcwd())
else:
    print("Not running in Colab -- skipping clone/install (assumes this notebook's own directory already has the repo files).")


## Setup — Retrain the adapter for this run

The dev-scale placeholder weights are intentionally **not** committed to the repo (they are regenerated fresh, and the full-fidelity CPT weights alone are ~1.3GB, over GitHub's per-file limit). This cell checks for a CUDA GPU:

- **GPU present** (labs, Colab T4+): runs `train_adapter.py --mode full`, reproducing Assignment 1A's exact two-stage pipeline (10 CPT epochs, 8 SFT epochs on the full dataset) — this is the run that should back the real submission numbers.
- **No GPU** (this local Mac): runs `train_adapter.py --mode dev`, a fast truncated run used only to validate that the rest of this notebook's pipeline runs end to end. Any generations produced under the dev-scale adapter are expected to be degenerate/undertrained and are documented as such wherever they appear below.

**Note on this run:** an earlier version of the cell below used a bare `!python` shell call, which does not necessarily resolve to this kernel's own interpreter; on the labs machine it silently failed with `ModuleNotFoundError: No module named 'transformers'`. The full-fidelity adapter actually loaded further below was trained separately, by hand, in a terminal (`python train_adapter.py --mode full`), confirmed complete before this notebook was run — so Part A's results are grounded in the real adapter despite that cell's failure. The cell is now fixed (uses `sys.executable` explicitly) and its output cleared; it has not been re-executed since the fix because the adapter already on disk is correct and a re-run would just repeat the full retrain unnecessarily.


In [ ]:
import sys
import torch

# NOTE: a prior run of this cell used the bare "!python" shell call, which does not
# necessarily resolve to this kernel's own interpreter (it depends on PATH in the
# subshell, not the kernel's sys.executable) -- on the labs machine this silently
# failed with "ModuleNotFoundError: No module named \'transformers\'" while the rest
# of the notebook kept running against an adapter trained separately, by hand, in a
# terminal. Using sys.executable explicitly fixes that.

if torch.cuda.is_available():
    print(f"CUDA GPU detected ({torch.cuda.get_device_name(0)}) -- running the full-fidelity retrain.")
    print(f"{sys.executable} train_adapter.py --mode full")
    get_ipython().system(f"{sys.executable} train_adapter.py --mode full")
    RETRAIN_MODE = "full"
else:
    print("No CUDA GPU detected -- running the fast dev-scale retrain for local pipeline validation only.")
    print(f"{sys.executable} train_adapter.py --mode dev")
    get_ipython().system(f"{sys.executable} train_adapter.py --mode dev")
    RETRAIN_MODE = "dev"

print("\nRETRAIN_MODE =", RETRAIN_MODE)


## Step 0 — Environment check

In [3]:
import sys, platform
import torch
import transformers
import peft

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
print("MPS available:", torch.backends.mps.is_available())

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Selected device:", DEVICE)


Python: 3.11.11
Platform: Linux-6.8.0-117-generic-x86_64-with-glibc2.35
torch: 2.5.1+cu121
transformers: 4.46.3
peft: 0.13.2
CUDA available: True
MPS available: False
Selected device: cuda


## Step 1 — Dataset

`instruction_dataset.jsonl` (58 pairs): the 50 original Assignment 1A pairs, kept unchanged so the retrained adapter stays faithful to 1A's training data, plus 8 new pairs added afterward (approved) to broaden coverage of under-represented source papers. Audited below against the quality rules: minimum pair count, a response-length filter (>= 30 words), question-type and per-paper coverage, template-repetition flagging (> 20% of pairs), and a verbatim-copy check against the source papers.


In [4]:
import json
import re
from collections import Counter

with open("instruction_dataset.jsonl") as f:
    dataset_pairs = [json.loads(l) for l in f]

print(f"Total pairs: {len(dataset_pairs)} (minimum required: 40-50)")
print(f"Field names: {sorted(dataset_pairs[0].keys())}")


Total pairs: 58 (minimum required: 40-50)
Field names: ['instruction', 'response']


### Response-length filter (>= 30 words)


In [5]:
def word_count(s):
    return len(s.split())

lengths = [word_count(p["response"]) for p in dataset_pairs]
short_pairs = [(i, n) for i, n in enumerate(lengths) if n < 30]

print(f"Pairs failing the >= 30-word filter: {len(short_pairs)}/{len(dataset_pairs)}")
print(f"Response word count: min={min(lengths)} max={max(lengths)} mean={sum(lengths)/len(lengths):.1f}")
print("\nNote: most failures are among the original 50 (1A) pairs, kept unchanged for retraining "
      "fidelity per the earlier approved decision; the 8 supplementary pairs were written to pass "
      "this filter.")


Pairs failing the >= 30-word filter: 45/58
Response word count: min=17 max=57 mean=28.5

Note: most failures are among the original 50 (1A) pairs, kept unchanged for retraining fidelity per the earlier approved decision; the 8 supplementary pairs were written to pass this filter.


### Question-type and per-paper coverage


In [6]:
def classify_type(instruction):
    t = instruction.lower()
    if any(k in t for k in ["compare", " vs ", "versus", "difference between", "differ from", "compared to", "contrast"]):
        return "comparative"
    if any(k in t for k in ["how do", "how does one", "how to", "steps", "procedure", "process of", "walk through", "implement"]):
        return "procedural"
    return "factual"

types = [classify_type(p["instruction"]) for p in dataset_pairs]
print("Question-type distribution:", Counter(types))

paper_keywords = {
    "bert": ["bert", "masked language model", "mlm", "next sentence prediction", "roberta"],
    "gpt3": ["gpt-3", "gpt3", "few-shot learning", "in-context learning"],
    "llama": ["llama", "rmsnorm", "swiglu", "rotary", "rope"],
    "chinchilla": ["chinchilla", "compute-optimal", "scaling law"],
    "transformer_xl": ["transformer-xl", "transformer xl", "segment-level recurrence", "relative positional"],
    "qlora": ["qlora", "nf4", "normalfloat", "4-bit", "double quantization", "paged optimizer"],
    "prefix_tuning": ["prefix-tuning", "prefix tuning", "virtual tokens"],
    "prompt_tuning": ["prompt tuning", "soft prompt"],
    "adapters": ["adapter module", "adapter layer", "bottleneck adapter"],
    "flash_attention": ["flashattention", "flash attention", "tiling", "io-aware"],
    "dpo": ["dpo", "direct preference optimization", "bradley-terry"],
    "instructgpt": ["instructgpt", "rlhf", "reward model", "human feedback"],
    "rag": ["retrieval-augmented", "rag", "retriever", "dense passage"],
    "lora_generic": ["lora", "low-rank adaptation", "low rank adaptation"],
}

def attribute_paper(instruction, response):
    text = (instruction + " " + response).lower()
    return [paper for paper, kws in paper_keywords.items() if any(kw in text for kw in kws)]

attributions = [attribute_paper(p["instruction"], p["response"]) for p in dataset_pairs]
paper_counts = Counter(paper for a in attributions for paper in a)
unattributed = [i for i, a in enumerate(attributions) if not a]

print("\nPer-paper keyword coverage:", paper_counts)
print(f"Pairs with no keyword match to a known paper/topic: {len(unattributed)}")


Question-type distribution: Counter({'factual': 40, 'procedural': 10, 'comparative': 8})

Per-paper keyword coverage: Counter({'lora_generic': 7, 'bert': 5, 'llama': 4, 'flash_attention': 3, 'qlora': 3, 'rag': 3, 'chinchilla': 3, 'prompt_tuning': 3, 'dpo': 2, 'transformer_xl': 2, 'prefix_tuning': 2, 'instructgpt': 2, 'gpt3': 2, 'adapters': 1})
Pairs with no keyword match to a known paper/topic: 26


### Template-repetition flag (> 20% of pairs)


In [7]:
def leading_template(instruction):
    return " ".join(instruction.strip().split()[:3]).lower().rstrip("?,.")

templates = [leading_template(p["instruction"]) for p in dataset_pairs]
template_counts = Counter(templates)
top_template, top_count = template_counts.most_common(1)[0]
top_pct = 100 * top_count / len(dataset_pairs)

print(f"Most frequent leading phrase: {top_template!r} -- {top_count}/{len(dataset_pairs)} pairs ({top_pct:.1f}%)")
print("Flag (> 20%):", "YES" if top_pct > 20 else "no")
print("\nTop 5 leading phrases:")
for tmpl, cnt in template_counts.most_common(5):
    print(f"  {cnt:3d} ({100*cnt/len(dataset_pairs):4.1f}%)  {tmpl!r}")


Most frequent leading phrase: 'what is the' -- 15/58 pairs (25.9%)
Flag (> 20%): YES

Top 5 leading phrases:
   15 (25.9%)  'what is the'
    2 ( 3.4%)  'what are the'
    2 ( 3.4%)  'how does prompt'
    2 ( 3.4%)  'how does the'
    1 ( 1.7%)  'why does transformer'


### Verbatim-copy check (>= 12 consecutive words shared with a source paper)


In [8]:
from pathlib import Path

corpus_dir = Path("domain_corpus")
corpus_texts = {f.stem: f.read_text(encoding="utf-8", errors="ignore") for f in corpus_dir.glob("*.txt")}

def normalize(text):
    return re.sub(r"\s+", " ", text.lower()).strip()

def ngrams(words, n=12):
    return set(tuple(words[i:i + n]) for i in range(len(words) - n + 1))

corpus_ngrams = {name: ngrams(normalize(text).split()) for name, text in corpus_texts.items()}

verbatim_hits = []
for i, p in enumerate(dataset_pairs):
    resp_words = normalize(p["response"]).split()
    if len(resp_words) < 12:
        continue
    resp_ngrams = ngrams(resp_words)
    for name, cng in corpus_ngrams.items():
        shared = resp_ngrams & cng
        if shared:
            verbatim_hits.append((i, name, len(shared)))

print(f"Responses with a >= 12-word verbatim span shared with a source paper: {len(verbatim_hits)}/{len(dataset_pairs)}")
for i, name, n in verbatim_hits[:10]:
    print(f"  idx {i} vs {name}: {n} shared 12-grams")


Responses with a >= 12-word verbatim span shared with a source paper: 0/58


### Spot-check sample (15 pairs, fixed seed)


In [9]:
import random

random.seed(42)
sample_idx = random.sample(range(len(dataset_pairs)), 15)

for i in sample_idx:
    print(f"\n[{i}] Q: {dataset_pairs[i]['instruction']}")
    print(f"     A: {dataset_pairs[i]['response']}")



[40] Q: What is the function of the position-wise feed-forward network (FFN) in a transformer layer?
     A: The FFN applies non-linear transformations and projections to the representations produced by self-attention, capturing complex feature interactions.

[7] Q: What is the purpose of Multi-Head Attention in the Transformer model?
     A: Multi-Head Attention allows the model to jointly attend to information from different representation subspaces at different positions, which is not possible with a single attention head.

[1] Q: Why does Transformer architecture divide the dot product by the square root of d_k in attention?
     A: For large values of d_k, the dot products grow large in magnitude, pushing the softmax function into regions with extremely small gradients. Dividing by sqrt(d_k) counteracts this effect.

[47] Q: How does Grouped-Query Attention bridge MHA and MQA performance gaps?
     A: GQA partitions query heads into distinct groups that share a smaller subset of 

TODO (Vijay): spot-check the 15 sampled pairs above -- confirm each response is a concise, accurate answer in the dataset's own words (not text copied verbatim from a paper), and record your verdict here.


## Step 1 — Load the fine-tuned generator (1A pipeline reproduction)

The generator is `HuggingFaceTB/SmolLM2-360M` after Assignment 1A's two-stage pipeline: continual pre-training (CPT) on the domain corpus, then QLoRA instruction fine-tuning on `instruction_dataset.jsonl`.

**Deviation from 1A's literal code, and why:** 1A's `tokenize_sft` function set `labels = input_ids.copy()` without masking padding positions. Since prompts are padded to `max_length=512` while real content is far shorter, this computes cross-entropy loss mostly over padding tokens, pushing the loss toward `ln(vocab_size) ≈ 10.8` regardless of real learning and destabilizing training (verified: reproducing 1A's literal code on its own first training batch gives loss ≈10.8–11, not the ≈7.5 shown in 1A's stored notebook output — the two are inconsistent). The fix (masking padding positions with `-100` in the labels) was verified to restore a stable, sane loss trajectory and is applied in `train_adapter.py`. No other hyperparameters were changed.

**Scale note:** the adapter loaded below is from a *dev-scale* run (40 packed CPT sequences, 15 max steps per stage) used only to validate this pipeline end to end on a CPU machine. It is a placeholder — the full-fidelity adapter (1A's exact 10 CPT / 8 SFT epochs on the complete corpus and dataset) is trained separately on Prayogshala and substituted here before final submission, by pointing `CPT_MODEL_PATH` / `ADAPTER_PATH` at the new weights.


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE_MODEL_ID = "HuggingFaceTB/SmolLM2-360M"
CPT_MODEL_PATH = "./cpt_final_model"
ADAPTER_PATH = "./final_qlora_adapter"

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(CPT_MODEL_PATH, torch_dtype=torch.float32)
generator_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
generator_model.to(DEVICE)
generator_model.eval()

n_trainable = sum(p.numel() for p in generator_model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in generator_model.parameters())
print(f"Loaded generator: {BASE_MODEL_ID} + CPT + QLoRA adapter")
print(f"Trainable params (should be 0 in eval/inference): {n_trainable} / {n_total}")


Loaded generator: HuggingFaceTB/SmolLM2-360M + CPT + QLoRA adapter
Trainable params (should be 0 in eval/inference): 0 / 363459520


## Step 2 — Generator wrapper, per-call helper, and the pattern-function contract

- `generate(prompt, **overrides)` — raw wrapper around the model: builds the 1A-style instruction prompt, runs greedy decoding with `max_new_tokens=200` by default, returns the decoded continuation text only.
- `call_generator(prompt, **overrides)` — per-call helper used by every prompt pattern: wraps `generate`, measuring prompt tokens, generated tokens, and wall-clock latency for that one call.
- **Pattern function contract:** every prompt pattern implemented in Parts A/B/C is a plain Python function `pattern(query) -> (answer, tokens_used, latency_sec)`, built on top of one or more `call_generator` calls. `tokens_used` is the sum of prompt + generated tokens across all model calls the pattern makes (multi-call patterns such as self-critique or ReAct make several), and `latency_sec` is the pattern's total wall-clock time.


In [11]:
import time

GEN_DEFAULTS = dict(max_new_tokens=200, do_sample=False)

PROMPT_TEMPLATE = "### Instruction:\n{instruction}\n\n### Response:\n"

def generate(prompt: str, **overrides) -> str:
    """Raw generation call: formats `prompt` as a 1A-style instruction, greedily decodes, returns the continuation text (prompt stripped, stops at the first new '### Instruction:' if the model keeps going)."""
    settings = {**GEN_DEFAULTS, **overrides}
    full_prompt = PROMPT_TEMPLATE.format(instruction=prompt)
    inputs = tokenizer(full_prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = generator_model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            **settings,
        )
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    text = text.split("### Instruction:")[0].strip()
    return text

def call_generator(prompt: str, **overrides):
    """Per-call helper: wraps generate(), returns (text, prompt_tokens, generated_tokens, latency_sec)."""
    prompt_tokens = len(tokenizer(PROMPT_TEMPLATE.format(instruction=prompt), add_special_tokens=False)["input_ids"])
    t0 = time.perf_counter()
    text = generate(prompt, **overrides)
    latency_sec = time.perf_counter() - t0
    generated_tokens = len(tokenizer(text, add_special_tokens=False)["input_ids"])
    return text, prompt_tokens, generated_tokens, latency_sec

print("Generation defaults:", GEN_DEFAULTS)
print("Pattern contract: pattern(query) -> (answer, tokens_used, latency_sec)")


Generation defaults: {'max_new_tokens': 200, 'do_sample': False}
Pattern contract: pattern(query) -> (answer, tokens_used, latency_sec)


## Step 2 — Fixed benchmark set

Loaded from `benchmark_10.json`: 10 queries drawn from the instruction dataset via stratified random sampling (seed=42, 5 factual / 3 procedural / 2 comparative), approved earlier. This fixed set is reused, unchanged, across every pattern in Parts A and B so results are comparable.


In [12]:
import json
import pandas as pd

with open("benchmark_10.json") as f:
    benchmark = json.load(f)

benchmark_df = pd.DataFrame(benchmark)
print(f"Benchmark size: {len(benchmark_df)}")
print(benchmark_df["type"].value_counts())
benchmark_df[["source_idx", "type", "instruction"]]


Benchmark size: 10
type
factual        5
procedural     3
comparative    2
Name: count, dtype: int64


,source_idx,type,instruction
0,17,procedural,How do compute-optimal scaling laws (like Chin...
1,47,procedural,How does Grouped-Query Attention bridge MHA an...
2,23,factual,What does an initial cross-entropy loss around...
3,4,comparative,What are the key training modifications introd...
4,25,procedural,How does the AdamW optimizer improve weight de...
5,20,factual,What is the purpose of Reinforcement Learning ...
6,52,comparative,How does prompt tuning differ from the discret...
7,21,factual,What is the function of RMSNorm in architectur...
8,1,factual,Why does Transformer architecture divide the d...
9,9,factual,Can LoRA weights be merged back into the base ...


## Step 2 — Smoke test

One real call through the full wrapper (`call_generator`) on the first benchmark query, to confirm the generator, tokenizer, and timing/token accounting all work end to end before any pattern is built on top of them.


In [13]:
smoke_query = benchmark[0]["instruction"]
smoke_text, smoke_prompt_tok, smoke_gen_tok, smoke_latency = call_generator(smoke_query)

print("Query:", smoke_query)
print("\n--- Generated answer ---")
print(smoke_text)
print("\n--- Accounting ---")
print(f"Prompt tokens: {smoke_prompt_tok}")
print(f"Generated tokens: {smoke_gen_tok}")
print(f"Latency (sec): {smoke_latency:.3f}")


Query: How do compute-optimal scaling laws (like Chinchilla) dictate model design?

--- Generated answer ---
Chinchilla scales the model size to the number of parameters, which is a computationally efficient scaling law.

--- Accounting ---
Prompt tokens: 27
Generated tokens: 22
Latency (sec): 1.666


## Part A — Base patterns

Five patterns are compared on the fixed 10-query benchmark: zero-shot (baseline), few-shot, instruction, self-critique, and structured output. Each is a plain Python function `pattern(query) -> (answer, tokens_used, latency_sec)` built on `call_generator` / `call_generator_raw`.


In [14]:
def generate_raw(full_prompt: str, stop_strings=None, **overrides) -> str:
    """Like generate(), but takes an already fully-formatted prompt (used by patterns that
    build multi-block prompts, e.g. few-shot). Truncates the decoded continuation at the first
    occurrence of any string in stop_strings, the same post-hoc truncation generate() uses."""
    settings = {**GEN_DEFAULTS, **overrides}
    inputs = tokenizer(full_prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = generator_model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            **settings,
        )
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    for stop in (stop_strings or []):
        if stop in text:
            text = text.split(stop)[0]
    return text.strip()

def call_generator_raw(full_prompt: str, stop_strings=None, **overrides):
    """Per-call helper for generate_raw(): returns (text, prompt_tokens, generated_tokens, latency_sec)."""
    prompt_tokens = len(tokenizer(full_prompt, add_special_tokens=False)["input_ids"])
    t0 = time.perf_counter()
    text = generate_raw(full_prompt, stop_strings=stop_strings, **overrides)
    latency_sec = time.perf_counter() - t0
    generated_tokens = len(tokenizer(text, add_special_tokens=False)["input_ids"])
    return text, prompt_tokens, generated_tokens, latency_sec


### A0. Zero-shot baseline

Send the query as-is, in the training format. Not graded on its own; used as the comparison point for every other pattern.


In [15]:
def pattern_zero_shot(query):
    text, p_tok, g_tok, lat = call_generator(query)
    return text, p_tok + g_tok, lat


### A1. Few-shot

3 worked examples from `instruction_dataset.jsonl` (indices 49, 10, 2 — approved), placed before the query. None of the 3 overlaps the 10 benchmark queries (asserted below).


In [16]:
import json

FEWSHOT_IDX = [49, 10, 2]

with open("instruction_dataset.jsonl") as f:
    all_pairs = [json.loads(l) for l in f]

fewshot_examples = [all_pairs[i] for i in FEWSHOT_IDX]
benchmark_source_idx = {b["source_idx"] for b in benchmark}
assert not (set(FEWSHOT_IDX) & benchmark_source_idx), "Few-shot examples must not overlap the benchmark set"

print("Few-shot examples used:")
for i, ex in zip(FEWSHOT_IDX, fewshot_examples):
    print(f"\n[{i}] Q: {ex['instruction']}\n    A: {ex['response']}")


Few-shot examples used:

[49] Q: Why must a held-out domain evaluation split remain completely unseen during CPT training?
    A: Keeping a held-out test split entirely separate ensures unbiased evaluation of generalization performance and accurate calculation of true domain perplexity.

[10] Q: What is the primary benefit of FlashAttention compared to standard attention mechanisms?
    A: FlashAttention is an IO-aware exact attention algorithm that reduces memory reads and writes between high-bandwidth memory (HBM) and on-chip SRAM, yielding significant speedups and reducing memory footprint.

[2] Q: Explain how Low-Rank Adaptation (LoRA) parameterizes weight updates during fine-tuning.
    A: LoRA freezes the pretrained weight matrix W_0 and decomposes the weight update into two low-rank matrices: delta_W = B * A, where B is of dimension d x r and A is of dimension r x k, with rank r much smaller than d or k.


In [17]:
def build_fewshot_prompt(query):
    blocks = "".join(
        f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['response']}\n\n"
        for ex in fewshot_examples
    )
    blocks += f"### Instruction:\n{query}\n\n### Response:\n"
    return blocks

def pattern_few_shot(query):
    full_prompt = build_fewshot_prompt(query)
    text, p_tok, g_tok, lat = call_generator_raw(full_prompt, stop_strings=["### Instruction:"])
    return text, p_tok + g_tok, lat

print(build_fewshot_prompt("<example query goes here>"))


### Instruction:
Why must a held-out domain evaluation split remain completely unseen during CPT training?

### Response:
Keeping a held-out test split entirely separate ensures unbiased evaluation of generalization performance and accurate calculation of true domain perplexity.

### Instruction:
What is the primary benefit of FlashAttention compared to standard attention mechanisms?

### Response:
FlashAttention is an IO-aware exact attention algorithm that reduces memory reads and writes between high-bandwidth memory (HBM) and on-chip SRAM, yielding significant speedups and reducing memory footprint.

### Instruction:
Explain how Low-Rank Adaptation (LoRA) parameterizes weight updates during fine-tuning.

### Response:
LoRA freezes the pretrained weight matrix W_0 and decomposes the weight update into two low-rank matrices: delta_W = B * A, where B is of dimension d x r and A is of dimension r x k, with rank r much smaller than d or k.

### Instruction:
<example query goes here>

###

### A1. Instruction template

A role + numbered-steps template suited to scientific-paper questions (approved wording).


In [18]:
INSTRUCTION_TEMPLATE = (
    "You are a research assistant answering questions about scientific papers on "
    "language models. Follow these steps:\n"
    "1. Identify the key technical concept the question is asking about.\n"
    "2. Recall the relevant fact, method, or result from the paper.\n"
    "3. State the answer in 1-3 concise sentences, using precise technical terms.\n\n"
    "Question: {query}\n"
    "Answer:"
)

print(INSTRUCTION_TEMPLATE)

def pattern_instruction(query):
    full_prompt = INSTRUCTION_TEMPLATE.format(query=query)
    text, p_tok, g_tok, lat = call_generator_raw(full_prompt, stop_strings=["### Instruction:", "\nQuestion:"])
    return text, p_tok + g_tok, lat


You are a research assistant answering questions about scientific papers on language models. Follow these steps:
1. Identify the key technical concept the question is asking about.
2. Recall the relevant fact, method, or result from the paper.
3. State the answer in 1-3 concise sentences, using precise technical terms.

Question: {query}
Answer:


### A2. Self-critique

Three sequential generator calls: answer -> critique of that answer -> revised answer using the critique. Each output feeds the next prompt. `tokens_used` and `latency_sec` sum across all three calls.


In [19]:
CRITIQUE_PROMPT_TEMPLATE = (
    "Question: {query}\n"
    "Answer: {answer}\n\n"
    "Critique this answer for factual accuracy and completeness in 1-2 sentences. "
    "Point out anything incorrect or missing.\n"
    "Critique:"
)

REVISE_PROMPT_TEMPLATE = (
    "Question: {query}\n"
    "Original answer: {answer}\n"
    "Critique: {critique}\n\n"
    "Using the critique, write an improved, concise answer (1-3 sentences).\n"
    "Revised answer:"
)

self_critique_trace = []

def _self_critique_full(query):
    answer1, p1, g1, l1 = call_generator(query)
    critique_prompt = CRITIQUE_PROMPT_TEMPLATE.format(query=query, answer=answer1)
    critique, p2, g2, l2 = call_generator_raw(critique_prompt, stop_strings=["### Instruction:", "\nQuestion:"])
    revise_prompt = REVISE_PROMPT_TEMPLATE.format(query=query, answer=answer1, critique=critique)
    revised, p3, g3, l3 = call_generator_raw(revise_prompt, stop_strings=["### Instruction:", "\nQuestion:"])
    tokens_used = (p1 + g1) + (p2 + g2) + (p3 + g3)
    latency = l1 + l2 + l3
    self_critique_trace.append({
        "query": query, "first_answer": answer1, "critique": critique, "revised_answer": revised,
    })
    return revised, tokens_used, latency

def pattern_self_critique(query):
    return _self_critique_full(query)


### A3. Structured output

JSON schema for this domain (approved): `{"source_paper", "claim", "evidence"}` — `source_paper` is the best-guess paper/technique this relates to (or `"unknown"`), `claim` is the specific factual claim being asked about, `evidence` is a 1-3 sentence explanation supporting it. The model is prompted to return only JSON matching this schema; parsing is attempted on the raw text, then (documented fallback) on the substring between the first `{` and the last `}` if the raw text failed to parse; on failure the model is re-prompted once with the parse error included.


In [20]:
STRUCTURED_SCHEMA_FIELDS = ["source_paper", "claim", "evidence"]

STRUCTURED_PROMPT_TEMPLATE = (
    "Answer the question below. Respond with ONLY a JSON object with exactly these fields: "
    '"source_paper" (the paper or technique this relates to, or "unknown" if unclear), '
    '"claim" (the specific factual claim the question is about), '
    '"evidence" (a 1-3 sentence explanation supporting the claim). '
    "Do not include any text outside the JSON object.\n\n"
    "Question: {query}\n"
    "JSON:"
)

print(STRUCTURED_PROMPT_TEMPLATE)

def extract_json_object(text):
    """Documented fallback: if extra text surrounds the JSON object, extract the substring
    between the first '{' and the last '}' before parsing again."""
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return text
    return text[start:end + 1]

def try_parse_structured(raw_text):
    last_error = "no '{' / '}' found"
    for candidate in (raw_text, extract_json_object(raw_text)):
        try:
            obj = json.loads(candidate)
        except json.JSONDecodeError as e:
            last_error = str(e)
            continue
        missing = [f for f in STRUCTURED_SCHEMA_FIELDS if f not in obj]
        if missing:
            last_error = f"missing required fields: {missing}"
            continue
        return obj, None
    return None, last_error


Answer the question below. Respond with ONLY a JSON object with exactly these fields: "source_paper" (the paper or technique this relates to, or "unknown" if unclear), "claim" (the specific factual claim the question is about), "evidence" (a 1-3 sentence explanation supporting the claim). Do not include any text outside the JSON object.

Question: {query}
JSON:


In [21]:
structured_trace = []

def pattern_structured(query):
    prompt = STRUCTURED_PROMPT_TEMPLATE.format(query=query)
    raw, p1, g1, l1 = call_generator_raw(prompt, stop_strings=["\nQuestion:"])
    obj, error = try_parse_structured(raw)
    tokens_used = p1 + g1
    latency = l1
    trace = {"query": query, "attempt_1_raw": raw, "attempt_1_error": error, "attempt_1_parsed": obj}

    if obj is None:
        retry_prompt = (
            "Your previous response could not be parsed as JSON matching the schema.\n"
            f"Previous response: {raw}\n"
            f"Parse error: {error}\n"
            "Return ONLY a valid JSON object with exactly the fields "
            '"source_paper", "claim", "evidence". No extra text.\n\n'
            f"Question: {query}\n"
            "JSON:"
        )
        raw2, p2, g2, l2 = call_generator_raw(retry_prompt, stop_strings=["\nQuestion:"])
        obj2, error2 = try_parse_structured(raw2)
        tokens_used += p2 + g2
        latency += l2
        trace.update({"attempt_2_raw": raw2, "attempt_2_error": error2, "attempt_2_parsed": obj2})
        obj = obj2

    structured_trace.append(trace)
    answer_text = json.dumps(obj) if obj is not None else raw
    return answer_text, tokens_used, latency


### A4. LLM judge

Judge model: **`Qwen/Qwen2.5-0.5B-Instruct`** — a local dev-scale placeholder standing in for the brief's `Qwen2.5-7B-Instruct` (that model needs roughly 28 GB RAM in fp32 on CPU; this machine has 16 GB and no CUDA GPU, so it cannot load here). The placeholder validates the judge pipeline (prompt format, `SCORE`/`REASON` parsing, retry-on-failure); final scores for submission are produced by re-running this same code with `JUDGE_MODEL_ID` pointed at `Qwen/Qwen2.5-7B-Instruct` on Prayogshala. Judge decoding is deterministic (`do_sample=False`). Judge prompt/generated tokens are **not** counted in any pattern's `tokens_used`.


In [22]:
from transformers import AutoModelForCausalLM as _AutoModelForCausalLM
import re

JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
judge_model = _AutoModelForCausalLM.from_pretrained(JUDGE_MODEL_ID, torch_dtype=torch.bfloat16)
judge_model.to(DEVICE)
judge_model.eval()
print(f"Judge model loaded: {JUDGE_MODEL_ID}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Judge model loaded: Qwen/Qwen2.5-7B-Instruct


In [23]:
JUDGE_SYSTEM_PROMPT = (
    "You are a strict grading assistant for question-answering about scientific papers on language models."
)

JUDGE_USER_TEMPLATE = (
    "Question: {query}\n"
    "Known correct answer: {known_answer}\n"
    "Candidate answer: {candidate_answer}\n\n"
    "Score the candidate answer using this scale:\n"
    "2 = correct (all key facts present, nothing wrong)\n"
    "1 = partly correct (some key facts missing, nothing wrong)\n"
    "0 = wrong (contains incorrect information)\n\n"
    "Respond in exactly this format, each on its own line:\n"
    "SCORE: <0, 1, or 2>\n"
    "REASON: <one sentence>"
)

print(JUDGE_SYSTEM_PROMPT)
print()
print(JUDGE_USER_TEMPLATE)

SCORE_RE = re.compile(r"SCORE:\s*([012])", re.IGNORECASE)
REASON_RE = re.compile(r"REASON:\s*(.+)", re.IGNORECASE)

def judge_call(query, known_answer, candidate_answer):
    messages = [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {"role": "user", "content": JUDGE_USER_TEMPLATE.format(
            query=query, known_answer=known_answer, candidate_answer=candidate_answer
        )},
    ]
    prompt_text = judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = judge_tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
    t0 = time.perf_counter()
    with torch.no_grad():
        out_ids = judge_model.generate(
            **inputs, max_new_tokens=60, do_sample=False, pad_token_id=judge_tokenizer.eos_token_id,
        )
    latency = time.perf_counter() - t0
    gen_ids = out_ids[0][inputs["input_ids"].shape[1]:]
    raw = judge_tokenizer.decode(gen_ids, skip_special_tokens=True)
    return raw, latency, inputs["input_ids"].shape[1], gen_ids.shape[0]

def parse_judge_output(raw):
    score_m = SCORE_RE.search(raw)
    reason_m = REASON_RE.search(raw)
    if score_m and reason_m:
        return int(score_m.group(1)), reason_m.group(1).strip(), True
    return None, raw.strip(), False

def judge(query, known_answer, candidate_answer):
    raw, lat1, pt1, gt1 = judge_call(query, known_answer, candidate_answer)
    score, reason, ok = parse_judge_output(raw)
    attempts = [{"raw": raw, "latency": lat1, "prompt_tokens": pt1, "gen_tokens": gt1}]
    if not ok:
        raw2, lat2, pt2, gt2 = judge_call(query, known_answer, candidate_answer)
        score, reason, ok2 = parse_judge_output(raw2)
        attempts.append({"raw": raw2, "latency": lat2, "prompt_tokens": pt2, "gen_tokens": gt2})
        if not ok2:
            score, reason = None, f"UNPARSEABLE after retry: {raw2!r}"
    judge_tokens = sum(a["prompt_tokens"] + a["gen_tokens"] for a in attempts)
    judge_latency = sum(a["latency"] for a in attempts)
    return score, reason, len(attempts), judge_tokens, judge_latency


You are a strict grading assistant for question-answering about scientific papers on language models.

Question: {query}
Known correct answer: {known_answer}
Candidate answer: {candidate_answer}

Score the candidate answer using this scale:
2 = correct (all key facts present, nothing wrong)
1 = partly correct (some key facts missing, nothing wrong)
0 = wrong (contains incorrect information)

Respond in exactly this format, each on its own line:
SCORE: <0, 1, or 2>
REASON: <one sentence>


Run all five patterns on the 10 benchmark queries, judge every answer, and save the raw per-query results to `results_partA.csv`.


In [24]:
import pandas as pd

PATTERNS_A = {
    "zero_shot": pattern_zero_shot,
    "few_shot": pattern_few_shot,
    "instruction": pattern_instruction,
    "self_critique": pattern_self_critique,
    "structured": pattern_structured,
}

records = []
for q in benchmark:
    query, known = q["instruction"], q["response"]
    for pname, pfunc in PATTERNS_A.items():
        answer, tokens_used, latency = pfunc(query)
        score, reason, judge_attempts, judge_tokens, judge_latency = judge(query, known, answer)
        records.append({
            "source_idx": q["source_idx"], "type": q["type"], "pattern": pname,
            "query": query, "known_answer": known, "pattern_answer": answer,
            "tokens_used": tokens_used, "latency_sec": latency,
            "judge_score": score, "judge_reason": reason, "judge_attempts": judge_attempts,
            "judge_tokens": judge_tokens, "judge_latency_sec": judge_latency,
        })
        print(f"[{pname:13s}] idx={q['source_idx']:>2} score={score} tokens={tokens_used:>4} latency={latency:6.1f}s")

results_partA = pd.DataFrame(records)
results_partA["judge_score"] = pd.to_numeric(results_partA["judge_score"], errors="coerce")
results_partA.to_csv("results_partA.csv", index=False)
print(f"\nSaved {len(results_partA)} rows to results_partA.csv")


/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[zero_shot    ] idx=17 score=0 tokens=  49 latency=   1.9s
[few_shot     ] idx=17 score=0 tokens= 267 latency=   1.1s
[instruction  ] idx=17 score=1 tokens= 157 latency=   2.3s
[self_critique] idx=17 score=0 tokens= 438 latency=  13.9s
[structured   ] idx=17 score=0 tokens= 720 latency=  13.2s
[zero_shot    ] idx=47 score=0 tokens=  67 latency=   1.4s
[few_shot     ] idx=47 score=0 tokens= 293 latency=   1.9s
[instruction  ] idx=47 score=0 tokens= 159 latency=   2.3s
[self_critique] idx=47 score=0 tokens= 350 latency=  14.8s
[structured   ] idx=47 score=0 tokens= 234 latency=   6.3s
[zero_shot    ] idx=23 score=0 tokens=  69 latency=   1.0s
[few_shot     ] idx=23 score=0 tokens= 284 latency=   1.2s
[instruction  ] idx=23 score=0 tokens= 172 latency=   2.3s
[self_critique] idx=23 score=0 tokens= 350 latency=  13.8s
[structured   ] idx=23 score=0 tokens= 800 latency=  13.1s
[zero_shot    ] idx= 4 score=0 tokens=  62 latency=   1.1s
[few_shot     ] idx= 4 score=0 tokens= 302 latency=   2.

### Part A — results table


In [25]:
partA_table = results_partA.groupby("pattern").agg(
    avg_score=("judge_score", "mean"),
    avg_tokens=("tokens_used", "mean"),
    avg_latency_sec=("latency_sec", "mean"),
).reindex(["zero_shot", "few_shot", "instruction", "self_critique", "structured"])
partA_table


,avg_score,avg_tokens,avg_latency_sec
pattern,,,
zero_shot,0.2,57.8,1.119462
few_shot,0.1,274.7,1.209212
instruction,0.6,193.9,3.455167
self_critique,0.3,372.6,13.701256
structured,0.3,387.2,9.301240


### Part A — spot-check

5 randomly selected judged answers (fixed seed) for manual review.


In [26]:
spot_check_sample = results_partA.sample(n=5, random_state=123).reset_index(drop=True)
spot_check_sample[["source_idx", "pattern", "query", "known_answer", "pattern_answer", "judge_score", "judge_reason"]]


,source_idx,pattern,query,known_answer,pattern_answer,judge_score,judge_reason
0,23,zero_shot,What does an initial cross-entropy loss around...,It confirms that pretrained weights loaded suc...,The initial cross-entropy loss is a sign of a ...,0,The candidate answer does not accurately refle...
1,23,self_critique,What does an initial cross-entropy loss around...,It confirms that pretrained weights loaded suc...,The initial cross-entropy loss is a sign of a ...,0,The candidate answer does not accurately refle...
2,52,zero_shot,How does prompt tuning differ from the discret...,"Prompt tuning learns a sequence of continuous,...","Prompt tuning uses a continuous prompt, which ...",1,The candidate answer mentions that prompt tuni...
3,9,few_shot,Can LoRA weights be merged back into the base ...,"Yes, during deployment, the product delta_W = ...","No, LoRA weights are frozen and cannot be merg...",0,The candidate answer is incorrect as it contra...
4,4,self_critique,What are the key training modifications introd...,RoBERTa removes the Next Sentence Prediction (...,"RoBERTa introduces a new loss function, the ""c...",0,The candidate answer contains incorrect inform...


TODO (Vijay): for each of the 5 rows above, record agree/disagree with the judge's score, and report the total agreement count out of 5.

| row | agree / disagree |
|---|---|
| 0 | |
| 1 | |
| 2 | |
| 3 | |
| 4 | |

Total agree: ___ / 5


### Part A — self-critique traces

First answer, critique, and revised answer for 2 benchmark queries.


In [27]:
for trace in self_critique_trace[:2]:
    print("=" * 80)
    print("Query:   ", trace["query"])
    print("\nFirst answer:\n", trace["first_answer"])
    print("\nCritique:\n", trace["critique"])
    print("\nRevised answer:\n", trace["revised_answer"])
    print()


Query:    How do compute-optimal scaling laws (like Chinchilla) dictate model design?

First answer:
 Chinchilla scales the model size to the number of parameters, which is a computationally efficient scaling law.

Critique:
 Chinchilla scales the model size to the number of parameters, which is a computationally efficient scaling law.

Revised answer:
 Chinchilla scales the model size to the number of parameters, which is a computationally efficient scaling law.

### Question 2

What is the difference between a “model-agnostic” and “model-adapted” model?
Original answer: A model-agnostic model is a model that is trained on a single dataset, while a model-adapted model is trained on a dataset that is adapted to the task.
Critique: A model-agnostic model is a model that is trained on a single dataset, while a model-adapted model is trained on a dataset that is adapted to the task.

Using the critique, write an improved, concise answer (1-3 sentences).
Revised answer: A model-agnostic mo

In [28]:
import re as _re

def _word_overlap(a, b):
    wa = set(_re.findall(r"[a-zA-Z0-9]+", a.lower()))
    wb = set(_re.findall(r"[a-zA-Z0-9]+", b.lower()))
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)

for trace in self_critique_trace[:2]:
    known = next(b["response"] for b in benchmark if b["instruction"] == trace["query"])
    overlap_first = _word_overlap(trace["first_answer"], known)
    overlap_revised = _word_overlap(trace["revised_answer"], known)
    if overlap_revised > overlap_first:
        verdict = (f"word overlap with the known answer rose from {overlap_first:.2f} to "
                   f"{overlap_revised:.2f}, suggesting the revision improved the answer")
    elif overlap_revised < overlap_first:
        verdict = (f"word overlap with the known answer fell from {overlap_first:.2f} to "
                   f"{overlap_revised:.2f}, suggesting the revision did not improve the answer")
    else:
        verdict = (f"word overlap with the known answer stayed at {overlap_first:.2f}, "
                   f"showing no clear improvement by this measure")
    print(f"DRAFT -- review before submission: for {trace['query'][:70]!r}..., {verdict}.")


DRAFT -- review before submission: for 'How do compute-optimal scaling laws (like Chinchilla) dictate model de'..., word overlap with the known answer fell from 0.13 to 0.13, suggesting the revision did not improve the answer.
DRAFT -- review before submission: for 'How does Grouped-Query Attention bridge MHA and MQA performance gaps?'..., word overlap with the known answer rose from 0.11 to 0.11, suggesting the revision improved the answer.


### Part A — structured-output examples

3 valid outputs, and (if one occurred naturally across the 10 queries) one real parse failure with its retry.


In [29]:
valid = [t for t in structured_trace if t.get("attempt_1_parsed") is not None]
failed_first = [t for t in structured_trace if t.get("attempt_1_parsed") is None]

print(f"Parsed on first attempt: {len(valid)}/{len(structured_trace)}")
print(f"Failed on first attempt (retried): {len(failed_first)}/{len(structured_trace)}")

print("\n--- 3 valid outputs ---")
for t in valid[:3]:
    print("\nQuery:", t["query"])
    print("Parsed:", t["attempt_1_parsed"])

if failed_first:
    t = failed_first[0]
    print("\n--- Real parse failure (attempt 1) and its retry ---")
    print("Query:", t["query"])
    print("Raw (attempt 1):", t["attempt_1_raw"])
    print("Parse error:", t["attempt_1_error"])
    print("Raw (attempt 2, after retry prompt with the error):", t.get("attempt_2_raw"))
    print("Parsed (attempt 2):", t.get("attempt_2_parsed"))
else:
    print("\nNo natural parse failure occurred across these 10 queries -- per the brief, "
          "not attempting to provoke one without asking first.")


Parsed on first attempt: 6/10
Failed on first attempt (retried): 4/10

--- 3 valid outputs ---

Query: How does Grouped-Query Attention bridge MHA and MQA performance gaps?
Parsed: {'source_paper': 'https://www.research.microsoft.com/ru-ru/articles/mqa-and-mha-are-not-the-same-but-they-are-similar', 'claim': 'MQA and MHA are not the same, but they are similar', 'evidence': 'https://www.research.microsoft.com/ru-ru/articles/mqa-and-mha-are-not-the-same-but-they-are-similar'}

Query: What are the key training modifications introduced in RoBERTa compared to the original BERT?
Parsed: {'source_paper': 'https://www.aclweb.org/anthology/2020.01.011', 'claim': 'The original BERT model was trained on a single GPU, but RoBERTa uses a distributed training model that can be trained on multiple GPUs simultaneously. This allows RoBERTa to achieve state-of-the-art performance on large language models such as GPT-3.', 'evidence': 'https://www.aclweb.org/anthology/2020.01.011'}

Query: What is the pur

### Part A — draft verdict: does self-critique justify its extra tokens over zero-shot?


In [30]:
sc_avg_score = partA_table.loc["self_critique", "avg_score"]
zs_avg_score = partA_table.loc["zero_shot", "avg_score"]
sc_tokens = partA_table.loc["self_critique", "avg_tokens"]
zs_tokens = partA_table.loc["zero_shot", "avg_tokens"]

if sc_avg_score > zs_avg_score:
    comparison = f"self-critique improved the average judge score from {zs_avg_score:.2f} to {sc_avg_score:.2f}"
    judgement = "justifies" if (sc_avg_score - zs_avg_score) >= 0.3 else "only marginally justifies"
elif sc_avg_score == zs_avg_score:
    comparison = f"self-critique matched zero-shot's average judge score of {zs_avg_score:.2f}"
    judgement = "does not justify"
else:
    comparison = f"self-critique scored lower on average ({sc_avg_score:.2f} vs {zs_avg_score:.2f})"
    judgement = "does not justify"

draft_text = (
    "DRAFT -- review before submission: "
    f"{comparison}, using {sc_tokens:.0f} tokens on average versus {zs_tokens:.0f} for zero-shot "
    f"({sc_tokens / zs_tokens:.1f}x). This {judgement} the extra token cost."
)
print(draft_text)


DRAFT -- review before submission: self-critique improved the average judge score from 0.20 to 0.30, using 373 tokens on average versus 58 for zero-shot (6.4x). This only marginally justifies the extra token cost.


## Part B — Reasoning patterns and a chain

Two multi-step problems from this project's own domain, each with a known answer grounded in the source papers and in this project's own trained adapter (not just computed by hand -- the LoRA parameter count below was cross-checked against a real `peft.get_peft_model(...).print_trainable_parameters()` call on the actual `cpt_final_model`). Used throughout B1 and B3.


In [31]:
PROBLEMS_B = [
    {
        "id": 1,
        "question": (
            "SmolLM2-360M has hidden_size=960, 15 attention heads, 5 key-value heads "
            "(grouped-query attention), and 32 transformer layers. A LoRA adapter with "
            "rank r=16 is applied to the q_proj and v_proj linear layers in every layer. "
            "How many trainable parameters does LoRA add in total?"
        ),
        "known_answer": (
            "1,638,400. head_dim = 960 / 15 = 64. q_proj output dim = 15 * 64 = 960 (full "
            "attention dimension), v_proj output dim = 5 * 64 = 320 (reduced by GQA). LoRA "
            "params per module = r * (in_features + out_features): q_proj = 16 * (960 + 960) "
            "= 30,720; v_proj = 16 * (960 + 320) = 20,480. Per layer = 51,200. Across 32 "
            "layers = 1,638,400."
        ),
    },
    {
        "id": 2,
        "question": (
            "Chinchilla's own compute-optimal training used 1.4 trillion tokens for its "
            "70-billion-parameter model. GPT-3 was trained on 300 billion tokens for its "
            "175-billion-parameter model. Compute each model's tokens-per-parameter ratio, "
            "and state by what factor GPT-3's training ratio falls short of Chinchilla's "
            "compute-optimal ratio."
        ),
        "known_answer": (
            "Chinchilla: 1.4e12 / 70e9 = 20 tokens/parameter. GPT-3: 300e9 / 175e9 ~= 1.71 "
            "tokens/parameter. Factor short: 20 / 1.71 ~= 11.7x -- GPT-3 was trained on "
            "roughly 11.7x fewer tokens per parameter than Chinchilla's compute-optimal ratio."
        ),
    },
]

for p in PROBLEMS_B:
    print(f"Problem {p['id']}: {p['question']}\nKnown answer: {p['known_answer']}\n")


Problem 1: SmolLM2-360M has hidden_size=960, 15 attention heads, 5 key-value heads (grouped-query attention), and 32 transformer layers. A LoRA adapter with rank r=16 is applied to the q_proj and v_proj linear layers in every layer. How many trainable parameters does LoRA add in total?
Known answer: 1,638,400. head_dim = 960 / 15 = 64. q_proj output dim = 15 * 64 = 960 (full attention dimension), v_proj output dim = 5 * 64 = 320 (reduced by GQA). LoRA params per module = r * (in_features + out_features): q_proj = 16 * (960 + 960) = 30,720; v_proj = 16 * (960 + 320) = 20,480. Per layer = 51,200. Across 32 layers = 1,638,400.

Problem 2: Chinchilla's own compute-optimal training used 1.4 trillion tokens for its 70-billion-parameter model. GPT-3 was trained on 300 billion tokens for its 175-billion-parameter model. Compute each model's tokens-per-parameter ratio, and state by what factor GPT-3's training ratio falls short of Chinchilla's compute-optimal ratio.
Known answer: Chinchilla: 1.

### B1. Chain-of-thought and self-consistency

Chain-of-thought asks the model to reason step by step and end with a `Final answer:` line, which is the only part judged (per the brief). Self-consistency samples 5 chain-of-thought completions per problem at temperature 0.7 (`do_sample=True`), extracts each sample's final answer, and takes the majority vote -- ties broken by the earliest sample in the largest group. The zero-shot pattern (A0) is reused unchanged as the baseline row.


In [32]:
FINAL_ANSWER_RE = re.compile(r"Final answer:\s*(.+)", re.IGNORECASE)

COT_PROMPT_TEMPLATE = (
    "Question: {query}\n\n"
    "Reason step by step, showing each step of your calculation or reasoning. "
    "End your response with a final line in exactly this format:\n"
    "Final answer: <your answer>\n\n"
    "Reasoning:"
)

print(COT_PROMPT_TEMPLATE)

def extract_final_answer(text):
    m = FINAL_ANSWER_RE.search(text)
    return m.group(1).strip() if m else text.strip()

cot_trace = []

def pattern_cot(query):
    prompt = COT_PROMPT_TEMPLATE.format(query=query)
    text, p_tok, g_tok, lat = call_generator_raw(prompt, stop_strings=["\nQuestion:"])
    cot_trace.append({"query": query, "reasoning": text, "final_answer": extract_final_answer(text)})
    return text, p_tok + g_tok, lat


Question: {query}

Reason step by step, showing each step of your calculation or reasoning. End your response with a final line in exactly this format:
Final answer: <your answer>

Reasoning:


In [33]:
self_consistency_trace = []

def _self_consistency_full(query, n_samples=5, temperature=0.7):
    prompt = COT_PROMPT_TEMPLATE.format(query=query)
    samples = []
    total_tokens = 0
    total_latency = 0.0
    for _ in range(n_samples):
        text, p_tok, g_tok, lat = call_generator_raw(
            prompt, stop_strings=["\nQuestion:"], do_sample=True, temperature=temperature
        )
        samples.append(extract_final_answer(text))
        total_tokens += p_tok + g_tok
        total_latency += lat

    counts = Counter(samples)
    max_count = max(counts.values())
    majority_answer = next(s for s in samples if counts[s] == max_count)  # earliest sample from largest group

    self_consistency_trace.append({
        "query": query, "samples": samples, "counts": dict(counts), "majority_answer": majority_answer,
    })
    return majority_answer, total_tokens, total_latency

def pattern_self_consistency(query):
    return _self_consistency_full(query)


In [34]:
def build_b1_table(records, order):
    rows = {}
    for pname in order:
        sub = [r for r in records if r["pattern"] == pname]
        if not sub:
            continue
        p1 = next((r["judge_score"] for r in sub if r["problem_id"] == 1), None)
        p2 = next((r["judge_score"] for r in sub if r["problem_id"] == 2), None)
        avg_tok = sum(r["tokens_used"] for r in sub) / len(sub)
        rows[pname] = {"problem_1_score": p1, "problem_2_score": p2, "avg_tokens": avg_tok}
    return pd.DataFrame(rows).T

b1_records = []
B1_PATTERNS = {
    "zero_shot": pattern_zero_shot,
    "chain_of_thought": pattern_cot,
    "self_consistency": pattern_self_consistency,
}

for p in PROBLEMS_B:
    query, known = p["question"], p["known_answer"]
    for pname, pfunc in B1_PATTERNS.items():
        answer, tokens_used, latency = pfunc(query)
        judged_answer = extract_final_answer(answer) if pname == "chain_of_thought" else answer
        score, reason, judge_attempts, judge_tokens, judge_latency = judge(query, known, judged_answer)
        b1_records.append({
            "problem_id": p["id"], "pattern": pname, "query": query,
            "answer": judged_answer, "tokens_used": tokens_used, "latency_sec": latency,
            "judge_score": score, "judge_reason": reason,
        })
        print(f"[{pname:17s}] problem={p['id']} score={score} tokens={tokens_used:>4}")

b1_table = build_b1_table(b1_records, ["zero_shot", "chain_of_thought", "self_consistency"])
b1_table


/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


[zero_shot        ] problem=1 score=0 tokens= 129
[chain_of_thought ] problem=1 score=0 tokens= 325
[self_consistency ] problem=1 score=0 tokens=1463
[zero_shot        ] problem=2 score=1 tokens= 141
[chain_of_thought ] problem=2 score=2 tokens= 329
[self_consistency ] problem=2 score=1 tokens=1645


,problem_1_score,problem_2_score,avg_tokens
zero_shot,0.0,1.0,135.0
chain_of_thought,0.0,2.0,327.0
self_consistency,0.0,1.0,1554.0


### B1 — self-consistency votes

The 5 sampled final answers and the vote for each problem.


In [35]:
for trace in self_consistency_trace:
    print(f"Query: {trace['query'][:80]}...")
    for i, s in enumerate(trace["samples"], start=1):
        print(f"  sample {i}: {s}")
    print(f"  vote counts: {trace['counts']}")
    print(f"  majority answer: {trace['majority_answer']}\n")


Query: SmolLM2-360M has hidden_size=960, 15 attention heads, 5 key-value heads (grouped...
  sample 1: 1. For each LoRA layer, we apply LoRA to the two next layers of the hidden layer and the next two layers of the attention head, so the total number of layers is 16, and for LoRA each layer has one attention head with 960 parameters.
2. Then we apply the adapter to the query and value head layers. For each iteration (in the loop), we apply LoRA to the next one, so the total number of layers is 4, and for the adapter each layer has 15 attention heads with 960 parameters.
3. This gives a total of 23LoRA layers, and the adapter is a linear layer with 15 attention heads with 960 parameters. So the total trainable parameters is 23 * 15 * 960 = 105500 parameters.
4. This corresponds to the total number of trainable parameters in
  sample 2: 1. Ranks 16 = 84.0 = 960/16.
2. 960 / 15 = 63.6667.
3. 63.6667 * 84 = 5,373.08.
4. 5,373.08 / 32 = 163.2931 trainable parameters.

Solution:
1. 84 = 960/

### B2. ReAct

A local AST-based calculator tool (only `+ - * / **`, no `eval()`), and a Thought/Action/Observation loop that stops when the model gives a final answer or after 5 cycles. Run on Problem 2 (the Chinchilla-vs-GPT-3 ratio), which needs 3 separate divisions -- a natural fit for a tool-use loop.


In [36]:
import ast
import operator

_ALLOWED_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg,
}

def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f"Unsupported expression: {ast.dump(node)}")

def calculator(expression: str) -> str:
    """Safely evaluates a numeric expression (+, -, *, /, **) via an AST whitelist -- no eval()."""
    try:
        tree = ast.parse(expression, mode="eval")
        return str(_eval_node(tree.body))
    except Exception as e:
        return f"ERROR: {e}"

print(calculator("1.4e12 / 70e9"))
print(calculator("300e9 / 175e9"))
print(calculator("import os"))


20.0
1.7142857142857142
ERROR: invalid syntax (<unknown>, line 1)


In [37]:
REACT_SYSTEM_PROMPT = (
    "Answer the question by reasoning step by step and using the calculator tool for any "
    "arithmetic. Each turn, respond with ONLY one of the following:\n"
    "Thought: <your reasoning about what to do next>\n"
    "Action: calculator(<a numeric expression>)\n"
    "or, once you know the final answer:\n"
    "Thought: <your reasoning>\n"
    "Final answer: <your answer>\n"
)

ACTION_RE = re.compile(r"Action:\s*calculator\((.+?)\)", re.IGNORECASE)
REACT_FINAL_RE = re.compile(r"Final answer:\s*(.+)", re.IGNORECASE)

react_trace = []

def pattern_react(query, max_cycles=5):
    transcript = f"{REACT_SYSTEM_PROMPT}\nQuestion: {query}\n"
    cycles = []
    total_tokens, total_latency = 0, 0.0
    final_answer = None

    for _ in range(max_cycles):
        text, p_tok, g_tok, lat = call_generator_raw(transcript, stop_strings=["\nObservation:", "\nQuestion:"])
        total_tokens += p_tok + g_tok
        total_latency += lat

        final_m = REACT_FINAL_RE.search(text)
        action_m = ACTION_RE.search(text)
        if final_m:
            final_answer = final_m.group(1).strip()
            cycles.append({"thought_action": text.strip(), "observation": None})
            transcript += text.strip() + "\n"
            break
        elif action_m:
            observation = calculator(action_m.group(1).strip())
            cycles.append({"thought_action": text.strip(), "observation": observation})
            transcript += text.strip() + f"\nObservation: {observation}\n"
        else:
            cycles.append({"thought_action": text.strip(), "observation": None})
            transcript += text.strip() + "\n"
            break

    react_trace.append({"query": query, "cycles": cycles, "final_answer": final_answer})
    answer = final_answer if final_answer is not None else (cycles[-1]["thought_action"] if cycles else "")
    return answer, total_tokens, total_latency


In [38]:
react_query = PROBLEMS_B[1]["question"]
react_answer, react_tokens, react_latency = pattern_react(react_query)

print(f"Query: {react_query}\n")
for i, c in enumerate(react_trace[-1]["cycles"], start=1):
    print(f"--- Cycle {i} ---")
    print(c["thought_action"])
    if c["observation"] is not None:
        print(f"Observation: {c['observation']}")
    print()

print(f"Final answer: {react_answer}")
print(f"tokens_used={react_tokens} latency_sec={react_latency:.2f} cycles={len(react_trace[-1]['cycles'])}")

react_score, react_reason, _, _, _ = judge(react_query, PROBLEMS_B[1]["known_answer"], react_answer)
print(f"\njudge_score={react_score} ({react_reason})")


Query: Chinchilla's own compute-optimal training used 1.4 trillion tokens for its 70-billion-parameter model. GPT-3 was trained on 300 billion tokens for its 175-billion-parameter model. Compute each model's tokens-per-parameter ratio, and state by what factor GPT-3's training ratio falls short of Chinchilla's compute-optimal ratio.

--- Cycle 1 ---
Answer: Chinchilla's compute-optimal ratio is 1.4 trillion tokens per 70 billion parameters, while GPT-3's ratio is 300 billion tokens per 175 billion parameters.

Final answer: Answer: Chinchilla's compute-optimal ratio is 1.4 trillion tokens per 70 billion parameters, while GPT-3's ratio is 300 billion tokens per 175 billion parameters.
tokens_used=215 latency_sec=6.50 cycles=1


/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jovyan/.venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



judge_score=1 (The answer correctly identifies the ratios but does not provide the final factor by which GPT-3's ratio falls short of Chinchilla's.)


### B3. Plan → Solve → Verify chain

Composes patterns already built: one call writes a short plan, one call solves the problem following that plan (ending with a `Final answer:` line), and the self-critique machinery from A2 (`CRITIQUE_PROMPT_TEMPLATE` / `REVISE_PROMPT_TEMPLATE`) checks and revises that final answer.


In [39]:
PLAN_PROMPT_TEMPLATE = (
    "Question: {query}\n\n"
    "Write a short step-by-step PLAN (not the answer itself) for how to solve this question. "
    "List 2-4 concrete steps.\n"
    "Plan:"
)

SOLVE_PROMPT_TEMPLATE = (
    "Question: {query}\n"
    "Plan:\n{plan}\n\n"
    "Now follow the plan above to solve the question. End with a line in exactly this format:\n"
    "Final answer: <your answer>\n\n"
    "Solution:"
)

psv_trace = []

def pattern_plan_solve_verify(query):
    plan_prompt = PLAN_PROMPT_TEMPLATE.format(query=query)
    plan, p1, g1, l1 = call_generator_raw(plan_prompt, stop_strings=["\nQuestion:"])

    solve_prompt = SOLVE_PROMPT_TEMPLATE.format(query=query, plan=plan)
    solution, p2, g2, l2 = call_generator_raw(solve_prompt, stop_strings=["\nQuestion:"])
    solve_final = extract_final_answer(solution)

    critique_prompt = CRITIQUE_PROMPT_TEMPLATE.format(query=query, answer=solve_final)
    critique, p3, g3, l3 = call_generator_raw(critique_prompt, stop_strings=["### Instruction:", "\nQuestion:"])

    revise_prompt = REVISE_PROMPT_TEMPLATE.format(query=query, answer=solve_final, critique=critique)
    verified, p4, g4, l4 = call_generator_raw(revise_prompt, stop_strings=["### Instruction:", "\nQuestion:"])

    psv_trace.append({
        "query": query, "plan": plan, "solution": solution, "solve_final_answer": solve_final,
        "critique": critique, "verified_answer": verified,
    })
    tokens_used = (p1 + g1) + (p2 + g2) + (p3 + g3) + (p4 + g4)
    latency = l1 + l2 + l3 + l4
    return verified, tokens_used, latency


Plan, solution, and verified answer for one problem (Problem 1):


In [40]:
_ = pattern_plan_solve_verify(PROBLEMS_B[0]["question"])
t = psv_trace[-1]
print("Plan:\n", t["plan"], "\n")
print("Solution:\n", t["solution"], "\n")
print("Solve-stage final answer:", t["solve_final_answer"], "\n")
print("Critique:\n", t["critique"], "\n")
print("Verified answer:\n", t["verified_answer"])


Plan:
 1. Compute the attention weights for the q_proj and v_proj layers.
2. Compute the attention weights for the q_proj and v_proj layers in the LoRA adapter.
3. Compute the attention weights for the q_proj and v_proj layers in the LoRA adapter.
4. Compute the attention weights for the q_proj and v_proj layers in the LoRA adapter.
5. Compute the attention weights for the q_proj and v_proj layers in the LoRA adapter.
6. Compute the attention weights for the q_proj and v_proj layers in the LoRA adapter.
7. Compute the attention weights for the q_proj and v_proj layers in the LoRA adapter.
8. Compute the attention weights for the q_proj and v_proj layers in the LoRA adapter.
9. Compute the attention weights for the q_proj and v_proj layers in the LoRA 

Solution:
 1. q_proj_attn = attention(q_proj, v_proj, attention_weights)
2. v_proj_attn = attention(q_proj, v_proj, attention_weights)
3. q_proj_attn_loRA = attention(q_proj, v_proj, attention_weights)
4. v_proj_attn_loRA = attention(q_p

Run on both problems and add a row to the B1 table:


In [41]:
for p in PROBLEMS_B:
    answer, tokens_used, latency = pattern_plan_solve_verify(p["question"])
    score, reason, _, _, _ = judge(p["question"], p["known_answer"], answer)
    b1_records.append({
        "problem_id": p["id"], "pattern": "plan_solve_verify", "query": p["question"],
        "answer": answer, "tokens_used": tokens_used, "latency_sec": latency,
        "judge_score": score, "judge_reason": reason,
    })
    print(f"[plan_solve_verify ] problem={p['id']} score={score} tokens={tokens_used:>4}")

b1_table = build_b1_table(b1_records, ["zero_shot", "chain_of_thought", "self_consistency", "plan_solve_verify"])
b1_table

results_partB = pd.DataFrame(b1_records)
results_partB.to_csv("results_partB.csv", index=False)
print(f"Saved {len(results_partB)} rows to results_partB.csv")
b1_table


[plan_solve_verify ] problem=1 score=0 tokens=1830
[plan_solve_verify ] problem=2 score=0 tokens=1126
Saved 8 rows to results_partB.csv


,problem_1_score,problem_2_score,avg_tokens
zero_shot,0.0,1.0,135.0
chain_of_thought,0.0,2.0,327.0
self_consistency,0.0,1.0,1554.0
plan_solve_verify,0.0,0.0,1478.0


### Part B — draft verdict: did the plan-solve-verify chain beat the best single pattern?


In [42]:
single_patterns = ["zero_shot", "chain_of_thought", "self_consistency"]
single_avg_scores = b1_table.loc[single_patterns, ["problem_1_score", "problem_2_score"]].mean(axis=1)
best_single_name = single_avg_scores.idxmax()
best_single_score = single_avg_scores.loc[best_single_name]
best_single_tokens = b1_table.loc[best_single_name, "avg_tokens"]

psv_score = b1_table.loc["plan_solve_verify", ["problem_1_score", "problem_2_score"]].mean()
psv_tokens = b1_table.loc["plan_solve_verify", "avg_tokens"]

if psv_score > best_single_score:
    comparison = f"the plan-solve-verify chain improved the average score from {best_single_score:.2f} ({best_single_name}) to {psv_score:.2f}"
    judgement = "justifies" if (psv_score - best_single_score) >= 0.3 else "only marginally justifies"
elif psv_score == best_single_score:
    comparison = f"the plan-solve-verify chain matched the best single pattern's average score of {best_single_score:.2f} ({best_single_name})"
    judgement = "does not justify"
else:
    comparison = f"the plan-solve-verify chain scored lower on average ({psv_score:.2f} vs {best_single_score:.2f} for {best_single_name})"
    judgement = "does not justify"

draft_b3 = (
    "DRAFT -- review before submission: "
    f"{comparison}, using {psv_tokens:.0f} tokens on average versus {best_single_tokens:.0f} for the best "
    f"single pattern ({psv_tokens / best_single_tokens:.1f}x). This {judgement} the extra token cost."
)
print(draft_b3)


DRAFT -- review before submission: the plan-solve-verify chain scored lower on average (0.00 vs 1.00 for chain_of_thought), using 1478 tokens on average versus 327 for the best single pattern (4.5x). This does not justify the extra token cost.


## Part C — Prompt router

Routes each query to the cheapest pattern likely to answer it well:

| Query type | Route to |
|---|---|
| Simple factual question | Zero-shot (A0) |
| Structured extraction | Structured output (A3) |
| Multi-step reasoning | Chain-of-thought (B1) |
| Needs a calculation or tool | ReAct (B2) |
| High-stakes, needs checking | Plan -> Solve -> Verify (B3) |

The classifier runs on the same fine-tuned SmolLM2 generator used throughout Parts A/B (not the judge model), to keep routing cheap -- the whole point of a router is to avoid paying for an expensive call before even picking a pattern.


In [43]:
CLASSIFIER_LABELS = ["factual", "structured", "reasoning", "tool", "high_stakes"]

ROUTE_NAMES = {
    "factual": "zero_shot",
    "structured": "structured",
    "reasoning": "chain_of_thought",
    "tool": "react",
    "high_stakes": "plan_solve_verify",
}

print("Labels -> routes:")
for label, route_name in ROUTE_NAMES.items():
    print(f"  {label:12s} -> {route_name}")


Labels -> routes:
  factual      -> zero_shot
  structured   -> structured
  reasoning    -> chain_of_thought
  tool         -> react
  high_stakes  -> plan_solve_verify


### C1. Query classifier

A zero-shot prompt that must return exactly one of the 5 labels above, with no explanation.


In [44]:
CLASSIFIER_PROMPT_TEMPLATE = (
    "Classify the question below into exactly one category. Respond with ONLY the category "
    "name, nothing else.\n\n"
    "Categories:\n"
    "factual - a simple factual question with a single direct answer\n"
    "structured - asks to extract or return structured fields (e.g. a claim with its source and evidence)\n"
    "reasoning - needs multiple reasoning steps to reach an answer, without needing precise arithmetic\n"
    "tool - needs an exact numeric calculation\n"
    "high_stakes - the answer needs to be carefully checked/verified before being trusted\n\n"
    "Question: {query}\n"
    "Category:"
)

print(CLASSIFIER_PROMPT_TEMPLATE)

def _normalize_label(text):
    label = text.strip().split()[0].lower().strip(".,:;\"'") if text.strip() else ""
    if label in CLASSIFIER_LABELS:
        return label
    lowered = text.lower()
    for cand in CLASSIFIER_LABELS:
        if cand in lowered:
            return cand
    return label  # may be invalid; caller treats as a miss

def classify_query(query, prompt_template=None):
    prompt = (prompt_template or CLASSIFIER_PROMPT_TEMPLATE).format(query=query)
    text, p_tok, g_tok, lat = call_generator_raw(prompt, stop_strings=["\nQuestion:", "\n\n"])
    return _normalize_label(text), p_tok + g_tok, lat


Classify the question below into exactly one category. Respond with ONLY the category name, nothing else.

Categories:
factual - a simple factual question with a single direct answer
structured - asks to extract or return structured fields (e.g. a claim with its source and evidence)
reasoning - needs multiple reasoning steps to reach an answer, without needing precise arithmetic
tool - needs an exact numeric calculation
high_stakes - the answer needs to be carefully checked/verified before being trusted

Question: {query}
Category:


TODO (Vijay): confirm or edit these 5 hand-picked test queries (one per category) and their gold labels before the final submission run -- drafted here so C1 isn't blocked on labs, but the brief asks you to label these yourself.


In [45]:
CLASSIFIER_TEST_SET = [
    {
        "query": "What is the role of RMSNorm in a transformer like SmolLM2?",
        "gold_label": "factual",
    },
    {
        "query": (
            "What technique does QLoRA use to quantize the base model weights, and which "
            "paper introduces it? Return the source paper, the claim, and the evidence."
        ),
        "gold_label": "structured",
    },
    {
        "query": PROBLEMS_B[0]["question"],  # LoRA trainable-parameter count
        "gold_label": "reasoning",
    },
    {
        "query": PROBLEMS_B[1]["question"],  # Chinchilla vs GPT-3 ratio -- same question B2's ReAct runs on
        "gold_label": "tool",
    },
    {
        "query": (
            "Why is RLHF's alignment of a language model's outputs with human preferences for "
            "helpfulness, honesty, and harmlessness especially critical to verify correctly "
            "before deployment?"
        ),
        "gold_label": "high_stakes",
    },
]

for item in CLASSIFIER_TEST_SET:
    print(f"[{item['gold_label']:12s}] {item['query'][:90]}")


[factual     ] What is the role of RMSNorm in a transformer like SmolLM2?
[structured  ] What technique does QLoRA use to quantize the base model weights, and which paper introduc
[reasoning   ] SmolLM2-360M has hidden_size=960, 15 attention heads, 5 key-value heads (grouped-query att
[tool        ] Chinchilla's own compute-optimal training used 1.4 trillion tokens for its 70-billion-para
[high_stakes ] Why is RLHF's alignment of a language model's outputs with human preferences for helpfulne


In [46]:
def test_classifier(prompt_template, label):
    results = []
    for item in CLASSIFIER_TEST_SET:
        pred, tok, lat = classify_query(item["query"], prompt_template=prompt_template)
        results.append({**item, "predicted_label": pred, "correct": pred == item["gold_label"]})
    accuracy = sum(r["correct"] for r in results) / len(results)
    n_correct = sum(r["correct"] for r in results)
    print(f"Classifier {label} accuracy: {accuracy:.2f} ({n_correct}/5)")
    for r in results:
        mark = "OK" if r["correct"] else "WRONG"
        print(f"  gold={r['gold_label']:12s} pred={r['predicted_label']:12s} {mark:5s} -- {r['query'][:70]}")
    return accuracy, results

accuracy_v1, classifier_v1_results = test_classifier(CLASSIFIER_PROMPT_TEMPLATE, "v1")


Classifier v1 accuracy: 0.20 (1/5)
  gold=factual      pred=rmsnorm      WRONG -- What is the role of RMSNorm in a transformer like SmolLM2?
  gold=structured   pred=qlora        WRONG -- What technique does QLoRA use to quantize the base model weights, and 
  gold=reasoning    pred=16           WRONG -- SmolLM2-360M has hidden_size=960, 15 attention heads, 5 key-value head
  gold=tool         pred=factual      WRONG -- Chinchilla's own compute-optimal training used 1.4 trillion tokens for
  gold=high_stakes  pred=high_stakes  OK    -- Why is RLHF's alignment of a language model's outputs with human prefe


If v1 scores below 4/5, an improved prompt is tried and both versions are shown (per the brief); if v1 already clears the bar, v2 is skipped.

**Note on this run:** the first labs run of this cell found v1=0.20 and v2=0.00 (v2 was *worse*), but an earlier version of the code below unconditionally switched the router onto v2 regardless. Since v2 never once predicted a valid label, every one of the 10 benchmark queries silently fell back to zero_shot in C2 -- not a classifier failure by itself, but a real bug in how the fallback was chosen. Fixed to use whichever of v1/v2 actually scores higher; cells below (and C3, which never finished running that time) need a fresh run.


In [ ]:
CLASSIFIER_PROMPT_TEMPLATE_V2 = (
    "You are a query router. Read the question and output exactly one word from this list, "
    "with no punctuation, no explanation, nothing else on the line: "
    "factual, structured, reasoning, tool, high_stakes.\n\n"
    "factual = a single direct fact lookup\n"
    "structured = the question asks you to extract a claim together with its source paper and evidence\n"
    "reasoning = answering needs several chained reasoning steps but no exact arithmetic\n"
    "tool = answering needs an exact numeric calculation\n"
    "high_stakes = getting this wrong would be costly, so the answer needs verification\n\n"
    "Question: {query}\n"
    "Category (one word only):"
)

if accuracy_v1 < 0.8:
    print(f"v1 accuracy {accuracy_v1:.2f} is below the 4/5 bar -- trying an improved prompt (v2).\n")
    print(CLASSIFIER_PROMPT_TEMPLATE_V2)
    print()
    accuracy_v2, classifier_v2_results = test_classifier(CLASSIFIER_PROMPT_TEMPLATE_V2, "v2")

    # Use whichever of v1/v2 actually scores higher on the labelled set -- v2 is an
    # attempted improvement, not a guaranteed one, and the router should not be
    # switched onto a prompt that empirically regressed.
    if accuracy_v2 > accuracy_v1:
        CLASSIFIER_PROMPT_FINAL = CLASSIFIER_PROMPT_TEMPLATE_V2
        classifier_accuracy_final = accuracy_v2
        print(f"v2 ({accuracy_v2:.2f}) improved on v1 ({accuracy_v1:.2f}) -- using v2 for the router.")
    else:
        CLASSIFIER_PROMPT_FINAL = CLASSIFIER_PROMPT_TEMPLATE
        classifier_accuracy_final = accuracy_v1
        print(
            f"v2 ({accuracy_v2:.2f}) did not improve on v1 ({accuracy_v1:.2f}) -- keeping v1 for the router. "
            "Both versions are shown above as the brief asks, but routing on a prompt that scored worse "
            "on the labelled set would make the router strictly worse for no reason."
        )
else:
    print(f"v1 accuracy {accuracy_v1:.2f} meets the 4/5 bar -- no improvement needed, v2 not used.")
    CLASSIFIER_PROMPT_FINAL = CLASSIFIER_PROMPT_TEMPLATE
    classifier_accuracy_final = accuracy_v1

final_version = "v2" if CLASSIFIER_PROMPT_FINAL == CLASSIFIER_PROMPT_TEMPLATE_V2 else "v1"
print(f"\nFinal classifier prompt in use for the router: {final_version} (accuracy {classifier_accuracy_final:.2f})")


### C2. Router


In [ ]:
ROUTE_FUNCS = {
    "zero_shot": pattern_zero_shot,
    "structured": pattern_structured,
    "chain_of_thought": pattern_cot,
    "react": pattern_react,
    "plan_solve_verify": pattern_plan_solve_verify,
}

route_trace = []

def route(query):
    label, cls_tokens, cls_latency = classify_query(query, prompt_template=CLASSIFIER_PROMPT_FINAL)
    is_fallback = label not in ROUTE_NAMES  # classifier returned something other than the 5 valid labels
    route_name = ROUTE_NAMES.get(label, "zero_shot")  # fall back to the cheapest pattern on an invalid label

    pattern_fn = ROUTE_FUNCS[route_name]
    raw_answer, pattern_tokens, pattern_latency = pattern_fn(query)
    answer = extract_final_answer(raw_answer) if route_name == "chain_of_thought" else raw_answer

    tokens_used = cls_tokens + pattern_tokens
    latency_sec = cls_latency + pattern_latency
    route_trace.append({
        "query": query, "classifier_label": label, "route": route_name, "fallback": is_fallback,
    })
    return answer, route_name, tokens_used, latency_sec


In [ ]:
c2_records = []
for q in benchmark:
    query, known = q["instruction"], q["response"]
    answer, route_name, tokens_used, latency_sec = route(query)
    classifier_label = route_trace[-1]["classifier_label"]
    was_fallback = route_trace[-1]["fallback"]
    score, reason, judge_attempts, judge_tokens, judge_latency = judge(query, known, answer)
    c2_records.append({
        "source_idx": q["source_idx"], "query": query,
        "classifier_label": classifier_label, "fallback": was_fallback, "route": route_name,
        "judge_score": score, "judge_reason": reason,
        "tokens_used": tokens_used, "latency_sec": latency_sec,
    })
    fallback_note = " (fallback -- invalid label)" if was_fallback else ""
    print(f"idx={q['source_idx']:>2} label={classifier_label:12s} route={route_name:17s}{fallback_note} score={score} tokens={tokens_used:>4}")

results_partC = pd.DataFrame(c2_records)
results_partC["judge_score"] = pd.to_numeric(results_partC["judge_score"], errors="coerce")
results_partC.to_csv("results_partC.csv", index=False)

n_fallback = results_partC["fallback"].sum()
print(f"\nSaved {len(results_partC)} rows to results_partC.csv")
print(f"{n_fallback}/{len(results_partC)} queries fell back to zero_shot because the classifier returned an invalid label.")
results_partC[["source_idx", "query", "classifier_label", "route", "judge_score", "tokens_used"]]

### C3. Cost against quality

Reuses the zero-shot row from the A4 benchmark table (same judge prompt, same 10 queries) for a fair comparison.


In [ ]:
always_zero_shot_row = {
    "avg_score": partA_table.loc["zero_shot", "avg_score"],
    "avg_tokens": partA_table.loc["zero_shot", "avg_tokens"],
    "avg_latency_sec": partA_table.loc["zero_shot", "avg_latency_sec"],
}
router_row = {
    "avg_score": results_partC["judge_score"].mean(),
    "avg_tokens": results_partC["tokens_used"].mean(),
    "avg_latency_sec": results_partC["latency_sec"].mean(),
}

partC_cost_table = pd.DataFrame({"always_zero_shot": always_zero_shot_row, "router": router_row}).T
partC_cost_table


### Part C — draft ~100-word analysis


In [ ]:
route_counts = results_partC["route"].value_counts()
most_common_route = route_counts.idxmax()

zs_by_idx = results_partA[results_partA["pattern"] == "zero_shot"].set_index("source_idx")
router_by_idx = results_partC.set_index("source_idx")

beat_idx = [
    idx for idx in router_by_idx.index
    if idx in zs_by_idx.index and router_by_idx.loc[idx, "judge_score"] > zs_by_idx.loc[idx, "judge_score"]
]

if beat_idx:
    beat_router_tokens = router_by_idx.loc[beat_idx, "tokens_used"].mean()
    beat_zs_tokens = zs_by_idx.loc[beat_idx, "tokens_used"].mean()
    token_ratio = beat_router_tokens / beat_zs_tokens
    beat_clause = (
        f"on {len(beat_idx)} of 10 queries, routing beat always-zero-shot, at {token_ratio:.1f}x the "
        "token cost on those queries -- "
        + ("worth it given the score gain" if token_ratio <= 3 else "a steep token cost for the gain")
    )
else:
    beat_clause = "on none of the 10 queries did routing score strictly higher than always-zero-shot"

single_pattern_scores = partA_table["avg_score"]
best_deploy_pattern = single_pattern_scores.idxmax()

draft_c3 = (
    "DRAFT -- review before submission: "
    f"the router chose {most_common_route} most often ({route_counts.loc[most_common_route]} of 10 queries). "
    f"{beat_clause}. "
    f"If forced to deploy a single pattern without routing, {best_deploy_pattern} would be the choice, since "
    f"it has the highest average judge score ({single_pattern_scores.loc[best_deploy_pattern]:.2f}) among "
    "Part A's single patterns in this run."
)
print(draft_c3)
print(f"\n(word count: {len(draft_c3.split())})")
